# New Roots Data Pipeline
## Bronze → Silver → Gold | Greater Melbourne

**Purpose:** Rebuild the New Roots suburb-level One Big Table (OBT), supporting language/infrastructure tables and map-ready features from the original source files.

**Workflow:** Raw source files (`Bronze/`) → cleaned and spatially matched Silver tables (`Silver/`) → application-ready Gold tables (`Gold/`).

**Run:** Start from a fresh kernel and select **Run All**. Paths are relative to this notebook's working directory. Refer to `README.md` before the first run.

> **Scope and interpretation:** Suburbs are selected using their representative point within the 2021 ABS Greater Melbourne boundary. Rental availability is **annual new rentals per 1,000 Census residents**, not vacancy rate. Scores are relative comparisons with percentile caps, not absolute measures of suburb quality.

## 0. Environment and setup
Install the required packages once in your chosen Python environment, then run the import/setup cell. Use `pip install -r requirements.txt` from a terminal if desired. The commented install command below is an optional fallback.

In [48]:
# Optional, run only when dependencies are missing:
# %pip install pandas geopandas openpyxl pyogrio scipy numpy shapely jupyter ipykernel

In [49]:
# Setup
from pathlib import Path
import zipfile
import re
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
import numpy as np
from difflib import SequenceMatcher

# Find the data folder from wherever this notebook is saved.
# VS Code provides the notebook's own path; Jupyter runs the notebook from its own folder, so the working directory works there.
notebook_dir = Path(globals().get("__vsc_ipynb_file__", Path.cwd())).resolve()
notebook_dir = notebook_dir.parent if notebook_dir.is_file() else notebook_dir

# Data lives one folder up from the pipeline folder, in "data" (Bronze, Silver and Gold sit inside it)
project = notebook_dir.parent / "data"
if not (project / "Bronze").is_dir():
    raise FileNotFoundError(f"Couldn't find {project / 'Bronze'}. Check the notebook is in the pipeline folder.")

print("Data folder:", project)
datasets = project / "Bronze"
cleaned = project / "Silver"
gold_dir = project / "Gold"
cleaned.mkdir(parents=True, exist_ok=True)
gold_dir.mkdir(parents=True, exist_ok=True)
if not datasets.is_dir():
    raise FileNotFoundError(f"Missing input folder: {datasets.resolve()}")

print("Datasets folder:", datasets.resolve())
print("Folder found:", datasets.exists())


def find_input(pattern):
    """Find a source file in Bronze, falling back to Silver (AMEP locations and the language display map are kept there)."""
    for folder in (datasets, cleaned):
        match = next(folder.rglob(pattern), None)
        if match is not None:
            return match
    raise FileNotFoundError(f"{pattern} not found in {datasets} or {cleaned}")


def normalize_suburb_name(name):
    """Lowercase a suburb name and drop any trailing bracketed tag, e.g. the
    "(Vic.)" ABS adds to names that also exist in another state. Used to line
    up suburb names across tables that don't follow the ABS convention."""
    name = re.sub(r"\s*\([^)]*\)\s*$", "", str(name))
    return name.strip().lower()

Data folder: C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data
Datasets folder: C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Bronze
Folder found: True


## 1. Bronze — load source data
The cells below load the original files without overwriting them. The ABS boundary, LGA and Vicmap shapefiles must include their accompanying `.shx`, `.dbf` and `.prj` files. File names and expected layouts are documented in the README.

In [50]:
# 1 - Median weekly rents
# The published workbook has multiple sheets and headings, so read them as they appear.
file = next(datasets.rglob("moving_annual_rent_latest.xlsx"))
ds_01 = pd.read_excel(file, sheet_name=None, header=None)

print("Rental sheets:", list(ds_01))

Rental sheets: ['1 bedroom flat', '2 bedroom flat', '3 bedroom flat', '2 bedroom house', '3 bedroom house', '4 bedroom house', 'All properties']


In [51]:
# 1b - LGA rental data and spatial boundaries

file = next(datasets.rglob("moving_annual_rent_latest_LGA.xlsx"))
ds_01_lga = pd.read_excel(file, sheet_name=None, header=None)

file = next(datasets.rglob("LGA_2021_AUST_GDA2020.shp"))
ds_lga = gpd.read_file(file)

print("LGA rental sheets:", list(ds_01_lga))
print("LGA spatial records:", len(ds_lga))

LGA rental sheets: ['1 bedroom flat', '2 bedroom flat', '3 bedroom flat', '2 bedroom house', '3 bedroom house', '4 bedroom house', 'All properties']
LGA spatial records: 566


In [52]:
# 1c - Affordable rental lettings

file = next(datasets.rglob("Affordable rental dwellings by Local Government Area*.xlsx"))

ds_affordable_lettings = pd.read_excel(file, sheet_name=None, header=None)

In [53]:
# 2 - Criminal incidents by suburb

file = next(datasets.rglob("*LGA*Criminal*.xlsx"))

ds_02 = pd.read_excel(file, sheet_name="Table 03")

print("Crime records:", len(ds_02))
display(ds_02.head())

Crime records: 341869


,Year,Year ending,Local Government Area,Postcode,Suburb/Town Name,Offence Division,Offence Subdivision,Offence Subgroup,Incidents Recorded
0,2026,June,Alpine,3691,Coral Bank,A Crimes against the person,A20 Assault and related offences,A211 FV Serious assault,1
1,2026,June,Alpine,3691,Coral Bank,B Property and deception offences,B20 Property damage,B21 Criminal damage,1
2,2026,June,Alpine,3691,Dederang,A Crimes against the person,"A70 Stalking, harassment and threatening behav...",A732 Non-FV Threatening behaviour,1
3,2026,June,Alpine,3691,Dederang,B Property and deception offences,B40 Theft,B42 Steal from a motor vehicle,1
4,2026,June,Alpine,3691,Dederang,D Public order and security offences,D10 Weapons and explosives offences,D11 Firearms offences,1


In [54]:
# 3 - ABS Census 2021 (population and median age)
# Continue reading G01/G02 from the original Census DataPack ZIP.
file = next(datasets.rglob("2021_GCP_SAL_VIC.zip"))
ds_03 = {}

with zipfile.ZipFile(file) as z:
    for name in z.namelist():
        filename = Path(name).name.upper()
        if filename.endswith(".CSV") and any(f"2021CENSUS_G{t}" in filename for t in ["01_", "02_"]):
            with z.open(name) as csv_file:
                ds_03[Path(name).stem] = pd.read_csv(csv_file, low_memory=False)

print("Census tables:", list(ds_03))
assert any("G01_" in k.upper() for k in ds_03), "Census G01 is missing"
assert any("G02_" in k.upper() for k in ds_03), "Census G02 is missing"

# Detailed language-used-at-home data exported from 2021 Census TableBuilder.
# Save the supplied CSV anywhere within the datasets folder.
language_file = next(datasets.rglob("lanuage by suburb.csv"))
ds_03_languages = pd.read_csv(language_file, skiprows=9, skipfooter=8, engine="python", encoding="utf-8-sig")
print("TableBuilder language columns:", len(ds_03_languages.columns))

Census tables: ['2021Census_G01_VIC_SAL', '2021Census_G02_VIC_SAL']
TableBuilder language columns: 507


In [55]:
# 4 - ABS suburb boundaries

file = next(datasets.rglob("SAL_2021*.shp"), None)

if file is not None:
    ds_04 = gpd.read_file(file)
else:
    file = next((datasets / "04_sal_2021_boundaries").glob("*.zip"))
    with zipfile.ZipFile(file) as z:
        shp = next(name for name in z.namelist() if name.lower().endswith(".shp"))
    ds_04 = gpd.read_file(f"/vsizip/{file.resolve().as_posix()}/{shp}")

print("Suburb boundaries:", len(ds_04))
display(ds_04.head(3))

Suburb boundaries: 15353


,SAL_CODE21,SAL_NAME21,STE_CODE21,STE_NAME21,AUS_CODE21,AUS_NAME21,AREASQKM21,LOCI_URI21,SHAPE_Leng,SHAPE_Area,geometry
0,10001,Aarons Pass,1,New South Wales,AUS,Australia,82.7639,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.554241,0.007975,"POLYGON ((149.82477 -32.84384, 149.83271 -32.8..."
1,10002,Abbotsbury,1,New South Wales,AUS,Australia,4.9788,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.123051,0.000485,"POLYGON ((150.86523 -33.88264, 150.86479 -33.8..."
2,10003,Abbotsford (NSW),1,New South Wales,AUS,Australia,1.0180,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.053423,0.000099,"POLYGON ((151.13472 -33.85492, 151.13445 -33.8..."


In [56]:
# 5 - Greater Melbourne boundary
# Shapefile 
file = next(datasets.rglob("*GCCSA*.shp"))
ds_05 = gpd.read_file(file)

print("Regions:", len(ds_05))
display(ds_05.head(3))

Regions: 35


,GCC_CODE21,GCC_NAME21,CHG_FLAG21,CHG_LBL21,STE_CODE21,STE_NAME21,AUS_CODE21,AUS_NAME21,AREASQKM21,LOCI_URI21,geometry
0,1GSYD,Greater Sydney,0,No change,1,New South Wales,AUS,Australia,12368.6858,http://linked.data.gov.au/dataset/asgsed3/GCCS...,"MULTIPOLYGON (((151.28159 -33.83318, 151.28151..."
1,1RNSW,Rest of NSW,0,No change,1,New South Wales,AUS,Australia,788428.9733,http://linked.data.gov.au/dataset/asgsed3/GCCS...,"MULTIPOLYGON (((159.0623 -31.50886, 159.06218 ..."
2,19499,No usual address (NSW),0,No change,1,New South Wales,AUS,Australia,NaN,http://linked.data.gov.au/dataset/asgsed3/GCCS...,None


In [57]:
# 6 - Vicmap Features of Interest
file = next(datasets.rglob("FOI_POINT.shp"))
ds_06 = gpd.read_file(file)

print("Vicmap locations:", len(ds_06))
display(ds_06.head(3))

Vicmap locations: 52118


,UFI,PFI,FEATURE_ID,PARENTFTID,FTYPE,FEATSUBTYP,FEATSTATUS,NAME,NAME_LABEL,PARENTNAME,...,VICNMSTATC,THEME1,THEME2,STATE,CRDATE_PFI,SUPER_PFI,CRDATE_UFI,FEATURE_UF,FEATURE_CR,geometry
0,45544976,984269,984269,984268.0,education centre,tertiary institution,None,CHISHOLM INSTITUTE OF TAFE - WONTHAGGI CAMPUS,Chisholm Institute Of Tafe - Wonthaggi Campus,CHISHOLM INSTITUTE OF TAFE,...,11,None,None,VIC,2012-08-24,NaN,2013-07-30,45544976,2013-07-30,MULTIPOINT ((145.58204 -38.59086))
1,45545434,136319,136319,NaN,education centre,tertiary institution,None,CATHEDRAL COLLEGE,Cathedral College,NaN,...,11,None,None,VIC,2009-05-21,NaN,2013-07-30,45545434,2013-07-30,MULTIPOINT ((146.32915 -36.3853))
2,45544409,634128,634128,634149.0,education centre,tertiary institution,None,GORDON INSTITUTE OF TAFE - EAST GEELONG CAMPUS,Gordon Institute Of Tafe - East Geelong Campus,GORDON INSTITUTE OF TAFE,...,11,None,None,VIC,2009-08-31,NaN,2013-07-30,45544409,2013-07-30,MULTIPOINT ((144.38308 -38.174))


In [58]:
# 7 - Public transport stops
# We do not need the much larger transport lines file for these metrics.
file = next(datasets.rglob("public_transport_stops.geojson"))
ds_07 = gpd.read_file(file)

print("Transport stops:", len(ds_07))
display(ds_07.head(3))

Transport stops: 32063


,STOP_ID,STOP_NAME,MODE,geometry
0,11212,Flinders Street Station,REGIONAL TRAIN,POINT (144.96627 -37.81809)
1,11213,Flinders Street Station,REGIONAL TRAIN,POINT (144.96649 -37.81814)
2,11214,Flinders Street Station,REGIONAL TRAIN,POINT (144.96652 -37.8182)


In [59]:
# 10 - Hospitals, GPs and pharmacies
# Each service type is stored in its own GeoJSON file.
health_input_folder = datasets / "10_health_services"
ds_10 = {}

for file in sorted(health_input_folder.glob("*.geojson")):
    ds_10[file.stem] = gpd.read_file(file)

assert ds_10, f"No GeoJSON files found in {health_input_folder}"
print("Health datasets:", list(ds_10))
for name, table in ds_10.items():
    print(name, len(table))

Health datasets: ['general_practices', 'hospitals', 'pharmacies']
general_practices 8082
hospitals 877
pharmacies 5622


In [60]:
# 11 - Adult Migrant English Program (AMEP) locations

file = find_input("amep_facilities.csv")

ds_11 = pd.read_csv(file)

print("AMEP locations:", len(ds_11))
display(ds_11.head())

AMEP locations: 145


,amep_id,facility_name,provider_name,street_address,suburb,postcode,state,latitude,longitude,phone,source,source_url
0,AMEP_VIC_001,Biyala Community Hub,Learning for Employment (LfE),2119 Warralily Boulevard,Armstrong Creek,3217,VIC,-38.240216,144.349019,03 5215 6270,Home Affairs (user supplied),https://immi.homeaffairs.gov.au/settling-in-au...
1,AMEP_VIC_002,Wingate Avenue Community Centre,Melbourne Polytechnic (Melbourne AMEP),13A Wingate Avenue,Ascot Vale,3032,VIC,-37.779499,144.919424,03 9212 0236,Home Affairs (user supplied),https://immi.homeaffairs.gov.au/settling-in-au...
2,AMEP_VIC_003,TAFE Gippsland,AMES Australia,32 Main Street,Bairnsdale,3875,VIC,-37.837927,147.596676,03 5120 4541,Home Affairs (user supplied),https://immi.homeaffairs.gov.au/settling-in-au...
3,AMEP_VIC_004,Djerriwarrh Community and Education Services,Learning for Employment (LfE),25-39 Barkly Street,Ballarat,3350,VIC,-37.563531,143.869437,03 9009 7900,Home Affairs (user supplied),https://immi.homeaffairs.gov.au/settling-in-au...
4,AMEP_VIC_005,Bellfield Community Hub,Melbourne Polytechnic (Melbourne AMEP),15 Daphne Crescent,Bellfield,3081,VIC,-37.754975,145.040132,03 9087 1155,Home Affairs (user supplied); Melbourne AMEP,https://immi.homeaffairs.gov.au/settling-in-au...


In [61]:
# 12 - Services Australia service centres
# The export has no geometry field, so read the properties and build points from latitude/longitude in Silver.

import json

file = find_input("Services_australia_geoJSON.json")

with open(file, encoding="utf-8") as f:
    ds_12 = pd.DataFrame([feature["properties"] for feature in json.load(f)["features"]])

print("Services Australia centres:", len(ds_12))
display(ds_12.head())

Services Australia centres: 59


,office_typ,address1,latitude,name,postcode,suburb,longitude
0,"Centrelink Service Centre - Medicare, Centrelink",,-37.758970,Heidelberg Service Centre,3084,Heidelberg,145.069725
1,"Centrelink Service Centre - Medicare, Centrelink",,-37.778159,Sunshine Service Centre,3020,Sunshine,144.829768
2,"Centrelink Service Centre - Medicare, Centrelink",,-37.757339,Lilydale Service Centre,3140,Lilydale,145.344086
3,"Centrelink Service Centre - Medicare, Centrelink",,-38.022550,Fountain Gate Service Centre,3805,Narre Warren,145.306310
4,"Centrelink Service Centre - Medicare, Centrelink",,-38.112460,Cranbourne Service Centre,3977,Cranbourne,145.284540


## 2. Silver — clean, align, match and enrich

Silver is the stable, auditable intermediate layer. Spatial overlays and distances are calculated in **EPSG:7855** (metres), then GeoDataFrames are exported as **EPSG:4326** WKT. LGA area weights are normalised within each suburb and reviewed before use in rent infill.

### 2.1 Suburb boundaries and area-weighted LGA relationships

In [62]:
# 4 & 5 - Suburbs in Greater Melbourne
# Keep suburb code, name, area and boundary for our future OBT.
melbourne = ds_05[ds_05["GCC_NAME21"].str.lower() == "greater melbourne"].to_crs(7855)
assert len(melbourne) == 1, "Expected one Greater Melbourne boundary"
melbourne_shape = melbourne.geometry.union_all()

vic_suburbs = ds_04[ds_04["STE_CODE21"].astype(str).str.lstrip("0") == "2"].to_crs(7855)
suburbs = vic_suburbs[vic_suburbs.geometry.representative_point().within(melbourne_shape)].copy()
suburbs = suburbs[["SAL_CODE21", "SAL_NAME21", "AREASQKM21", "geometry"]]
suburbs = suburbs.rename(columns={"SAL_CODE21": "suburb_id", "SAL_NAME21": "suburb_name", "AREASQKM21": "suburb_area_km2"})
suburbs["suburb_id"] = suburbs["suburb_id"].astype(str)
suburbs["suburb_area_km2"] = pd.to_numeric(suburbs["suburb_area_km2"], errors="coerce")

# normalised name for joining to crime/rental later, since they don't use ABS's "(Vic.)" tag
suburbs["suburb_name_join"] = suburbs["suburb_name"].map(normalize_suburb_name)
suburbs = suburbs.to_crs(4326).reset_index(drop=True)

print("Greater Melbourne suburbs:", len(suburbs))
display(suburbs.head())

# JG observation: ABS tags suburb names that also exist in another state with "(Vic.)",
# e.g. Abbotsford (Vic.). Nothing else in our other sources does this, so a plain name
# join against crime/rental would miss most suburbs. Added suburb_name_join above to fix that.

Greater Melbourne suburbs: 572


,suburb_id,suburb_name,suburb_area_km2,geometry,suburb_name_join
0,20002,Abbotsford (Vic.),1.7405,"POLYGON ((145.00195 -37.79665, 145.0019 -37.79...",abbotsford
1,20003,Aberfeldie,1.5515,"POLYGON ((144.89576 -37.76514, 144.89547 -37.7...",aberfeldie
2,20011,Aintree,6.7302,"POLYGON ((144.67179 -37.72628, 144.67022 -37.7...",aintree
3,20015,Airport West,3.6748,"POLYGON ((144.87979 -37.71565, 144.88016 -37.7...",airport west
4,20017,Albanvale,1.8634,"POLYGON ((144.76926 -37.74082, 144.77088 -37.7...",albanvale


In [63]:
# 4b - Suburb to LGA mapping

MATCH_THRESHOLD = 90
NAME_OVERRIDES = {"Moreland": "Merri-bek"}

def clean_lga_name(name):
    """Strip ABS administrative suffixes and punctuation for fuzzy LGA matching."""
    name = str(name).lower().replace("&", "and")
    name = re.sub(r"\([^)]*\)", " ", name)
    name = re.sub(r"\b(rural city|city|shire|borough|of)\b", " ", name)
    return re.sub(r"[^a-z0-9]", "", name)


# 1. Prepare Greater Melbourne suburb polygons
suburb_polygons = suburbs[["suburb_id", "suburb_name", "geometry"]].to_crs(7855).copy()

# 2. Prepare Victorian LGA polygons
lga_polygons = ds_lga.copy()

name_col = next((c for c in ["LGA_NAME21", "LGA_NAME", "lga_name"] if c in lga_polygons.columns), None)
if name_col is None:
    raise ValueError("No LGA name column found")

if "LGA_CODE21" in lga_polygons.columns:
    lga_polygons = lga_polygons[lga_polygons["LGA_CODE21"].astype(str).str.startswith("2")].copy()

lga_polygons = lga_polygons[lga_polygons[name_col].notna() & ~lga_polygons[name_col].str.contains("Unincorporated", case=False, na=False)].copy()

lga_polygons = lga_polygons.rename(columns={name_col: "lga_spatial_name"})[["lga_spatial_name", "geometry"]].to_crs(7855)

# 3. Calculate overlaps before matching LGA names
overlaps = gpd.overlay(suburb_polygons, lga_polygons, how="intersection", keep_geom_type=True)

overlaps["overlap_area"] = overlaps.geometry.area
overlaps = overlaps[overlaps["overlap_area"] > 0].copy()

# Only retain LGAs overlapping included suburbs
lga_polygons = lga_polygons[lga_polygons["lga_spatial_name"].isin(overlaps["lga_spatial_name"])].copy()

print("Relevant LGAs:", len(lga_polygons))

# 4. Extract LGA names from rental workbook
rental_sheet = ds_01_lga["1 bedroom flat"]

excluded = {"Group Total", "Total", "Victoria", "Metropolitan Melbourne", "Regional Victoria"}

rental_names = sorted(set(rental_sheet.iloc[3:, 1].dropna().astype(str).str.strip()) - excluded - {""})

# 5. Fuzzy name matching
def match_lga_name(spatial_name):
    """Return best rent-workbook name, similarity (0–100), and review status."""
    if spatial_name in NAME_OVERRIDES:
        candidate = NAME_OVERRIDES[spatial_name]
        if candidate not in rental_names:
            raise ValueError(f"Missing rental LGA: {candidate}")
        return candidate, 100.0, "MANUAL OVERRIDE"

    best = max(rental_names, key=lambda x: SequenceMatcher(None, clean_lga_name(spatial_name), clean_lga_name(x)).ratio())

    score = round(100 * SequenceMatcher(None, clean_lga_name(spatial_name), clean_lga_name(best)).ratio(), 1)

    status = "MATCH" if score >= MATCH_THRESHOLD else "REVIEW"
    return best, score, status


lga_matches = lga_polygons[["lga_spatial_name"]].drop_duplicates().copy()

lga_matches[["lga_name", "match_score", "match_status"]] = pd.DataFrame(lga_matches["lga_spatial_name"].map(match_lga_name).tolist(), index=lga_matches.index)

# Flag duplicated rental names
lga_matches.loc[lga_matches["lga_name"].duplicated(keep=False), "match_status"] = "REVIEW DUPLICATE"

display(lga_matches.sort_values(["match_status", "match_score"]))

# Stop if any matches need manual review
if lga_matches["match_status"].str.startswith("REVIEW").any():
    raise ValueError("Review flagged LGA name matches and update NAME_OVERRIDES")

# 6. Apply matched rental names to suburb overlaps
joined = overlaps.merge(lga_matches[["lga_spatial_name", "lga_name"]], on="lga_spatial_name", how="left", validate="many_to_one")

# Calculate proportion of each suburb within each LGA
joined["lga_weighting"] = (joined["overlap_area"] / joined.groupby("suburb_id")["overlap_area"].transform("sum"))

# Final Silver mapping table
suburb_lga_mapping = joined[["suburb_id", "suburb_name", "lga_name", "lga_weighting"]].sort_values(["suburb_name", "lga_weighting"], ascending=[True, False]).reset_index(drop=True)

# 7. Data quality checks
weight_sums = suburb_lga_mapping.groupby("suburb_id")["lga_weighting"].sum()

assert np.isclose(weight_sums, 1, atol=0.001).all(), ("LGA weights do not sum to 1")

missing = sorted(set(suburbs["suburb_id"]) - set(suburb_lga_mapping["suburb_id"]))

print("Mapped suburbs:", suburb_lga_mapping["suburb_id"].nunique())
print("Mapping records:", len(suburb_lga_mapping))
print("Unmapped suburbs:", len(missing))

if missing:
    print("WARNING: Unmapped suburb IDs:", missing)

display(suburb_lga_mapping.head())

# 8. Clean LGA spatial data for Silver
lga_spatial_cleaned = lga_polygons.merge(lga_matches[["lga_spatial_name", "lga_name"]], on="lga_spatial_name", how="left", validate="many_to_one").to_crs(7844)

Relevant LGAs: 39


,lga_spatial_name,lga_name,match_score,match_status
182,Moreland,Merri-bek,100.0,MANUAL OVERRIDE
183,Mornington Peninsula,Mornington Penin'a,91.4,MATCH
134,Banyule,Banyule,100.0,MATCH
135,Bass Coast,Bass Coast,100.0,MATCH
136,Baw Baw,Baw Baw,100.0,MATCH
137,Bayside (Vic.),Bayside,100.0,MATCH
139,Boroondara,Boroondara,100.0,MATCH
140,Brimbank,Brimbank,100.0,MATCH
143,Cardinia,Cardinia,100.0,MATCH
144,Casey,Casey,100.0,MATCH


Mapped suburbs: 572
Mapping records: 1053
Unmapped suburbs: 0


,suburb_id,suburb_name,lga_name,lga_weighting
0,20002,Abbotsford (Vic.),Yarra,1.000000e+00
1,20002,Abbotsford (Vic.),Boroondara,3.519497e-13
2,20003,Aberfeldie,Moonee Valley,1.000000e+00
3,20003,Aberfeldie,Maribyrnong,5.368377e-14
4,20011,Aintree,Melton,1.000000e+00


### 2.2 Crime and Census demographics

In [64]:
# 2 - Clean and transform crime data

# Keep the latest year
crime = ds_02[ds_02["Year"] == 2026].copy()

# Keep only the columns needed
crime = crime[["Local Government Area", "Postcode", "Suburb/Town Name", "Offence Division", "Incidents Recorded"]]

# Rename columns
crime.columns = ["lga", "postcode", "suburb", "offence_type", "incidents"]

# Convert incident counts to numbers
crime["incidents"] = pd.to_numeric(crime["incidents"], errors="coerce")

# Separate crimes against people and property offences
crime["person_incidents"] = crime["incidents"].where(crime["offence_type"].str.startswith("A "), 0)

crime["property_incidents"] = crime["incidents"].where(crime["offence_type"].str.startswith("B "), 0)

# Combine records into one row per suburb, postcode and LGA
crime_clean = crime.groupby(["lga", "postcode", "suburb"], as_index=False)[["incidents", "person_incidents", "property_incidents"]].sum()

# Rename the total incidents column
crime_clean = crime_clean.rename(columns={"incidents": "total_incidents"})

# matching key for suburbs.suburb_name_join
crime_clean["suburb_join"] = crime_clean["suburb"].map(normalize_suburb_name)

# Crime data is reported statewide, not just for Melbourne, so drop anything that
# isn't one of our 572 suburbs - same scoping the suburbs/vicmap/transport/health
# tables already apply, just done by name instead of by shape.
crime_clean = crime_clean[crime_clean["suburb_join"].isin(suburbs["suburb_name_join"])]

print("Crime records:", len(crime_clean))
display(crime_clean.head())

# JG observation: joining straight on suburb name only matched ~20% of our suburbs. Turned out
# to be the ABS "(Vic.)" tag again - once that's stripped (see suburb_join), 569 of our 572
# Melbourne suburbs match. The remaining 3 just don't seem to have any recorded incidents.
# Also dropped every row that isn't one of our 572 suburbs - this data covers the whole
# state, so most of it was for towns well outside Melbourne and not relevant to the OBT.

Crime records: 648


,lga,postcode,suburb,total_incidents,person_incidents,property_incidents,suburb_join
113,Banyule,3079,Ivanhoe,922,114,697,ivanhoe
114,Banyule,3079,Ivanhoe East,123,5,96,ivanhoe east
115,Banyule,3081,Bellfield,127,20,86,bellfield
116,Banyule,3081,Heidelberg Heights,553,94,343,heidelberg heights
117,Banyule,3081,Heidelberg West,998,157,612,heidelberg west


In [65]:
# 3 - Census population and median age
# G01 and G02 have one row per suburb. Give the fields shorter names.
g01 = next(table for name, table in ds_03.items() if "G01_" in name.upper())
g02 = next(table for name, table in ds_03.items() if "G02_" in name.upper())

code_1 = next(c for c in g01.columns if "SAL_CODE" in c.upper())
code_2 = next(c for c in g02.columns if "SAL_CODE" in c.upper())
pop_col = "Tot_P_P"
age_col = next(c for c in g02.columns if "med" in c.lower() and "age" in c.lower())

population = g01[[code_1, pop_col]].rename(columns={code_1: "suburb_id", pop_col: "population"})
ages = g02[[code_2, age_col]].rename(columns={code_2: "suburb_id", age_col: "median_age"})

# ABS census codes come through as e.g. SAL20001 - strip the letters so they match
# the plain numeric codes used in the suburb boundary file
population["suburb_id"] = population["suburb_id"].astype(str).str.replace(r"^\D+", "", regex=True)
ages["suburb_id"] = ages["suburb_id"].astype(str).str.replace(r"^\D+", "", regex=True)
census_people = population.merge(ages, on="suburb_id", how="left")
census_people["population"] = pd.to_numeric(census_people["population"], errors="coerce")
census_people["median_age"] = pd.to_numeric(census_people["median_age"], errors="coerce")
census_people = census_people[census_people["suburb_id"].isin(suburbs["suburb_id"])]

print("Median age source column:", age_col)
display(census_people.head())

# JG observation: this came back empty at first - the census suburb codes have a "SAL" prefix
# that the boundary file's codes don't, so they never matched until stripped.

Median age source column: Median_age_persons


,suburb_id,population,median_age
1,20002,9088,33
2,20003,3925,41
10,20011,7982,30
14,20015,8173,39
16,20017,5641,37


In [66]:
# 3 - Census language used at home (2021 TableBuilder, detailed LANP)
# The TableBuilder CSV is wide: one suburb per row and one column per language.
language_wide = ds_03_languages.copy()
language_wide = language_wide.rename(columns={language_wide.columns[0]: "suburb_name"})
language_wide["suburb_name"] = language_wide["suburb_name"].astype("string").str.strip()

# Remove the extra header, state total, non-resident records, and unused columns.
language_wide = language_wide[language_wide["suburb_name"].notna() & ~language_wide["suburb_name"].isin(["SAL (UR)", "Total", "No usual address (Vic.)", "Migratory - Offshore - Shipping (Vic.)"])].copy()
language_columns = [c for c in language_wide.columns if c not in {"suburb_name", "Total", "Not stated"} and not c.startswith("Unnamed:")]

# Keep detailed named languages as well as explicitly labelled nfd/nec categories.
# Reject missing/invalid counts rather than silently treating them as zero.
language_numeric = language_wide[language_columns].apply(pd.to_numeric, errors="coerce")
language_wide = pd.concat([language_wide[["suburb_name"]], language_numeric], axis=1)
if language_wide[language_columns].isna().any().any():
    raise ValueError("TableBuilder language counts contain blank or nonnumeric values")
if language_wide[language_columns].lt(0).any().any():
    raise ValueError("TableBuilder language counts contain negative values")
if language_wide["suburb_name"].duplicated().any():
    raise ValueError("TableBuilder has duplicate suburb names")

# Join to Greater Melbourne suburbs: use exact ABS names first so places with
# the same normalised name cannot accidentally be assigned the wrong suburb ID.
suburb_lookup = suburbs[["suburb_id", "suburb_name", "suburb_name_join"]].copy()
if suburb_lookup["suburb_name"].duplicated().any():
    raise ValueError("Suburbs table has duplicate suburb names")
language_wide["suburb_name_join"] = language_wide["suburb_name"].map(normalize_suburb_name)
matched_exact = language_wide.merge(suburb_lookup[["suburb_id", "suburb_name"]], on="suburb_name", how="inner", validate="one_to_one")

# For names with different formatting, use the existing name normalisation,
# but only if the normalised name is unique in BOTH tables.
source_left = language_wide[~language_wide["suburb_name"].isin(matched_exact["suburb_name"])]
suburbs_left = suburb_lookup[~suburb_lookup["suburb_id"].isin(matched_exact["suburb_id"])]
source_left = source_left[~source_left["suburb_name_join"].duplicated(keep=False)]
suburbs_left = suburbs_left[~suburbs_left["suburb_name_join"].duplicated(keep=False)]
matched_fallback = source_left.merge(suburbs_left[["suburb_id", "suburb_name_join"]], on="suburb_name_join", how="inner", validate="one_to_one")
matched = pd.concat([matched_exact, matched_fallback], ignore_index=True)
if matched["suburb_id"].duplicated().any():
    raise ValueError("More than one language record matched the same suburb ID")

unmatched_suburbs = suburb_lookup[~suburb_lookup["suburb_id"].isin(matched["suburb_id"])]
print(f"TableBuilder suburbs matched: {len(matched)} / {len(suburbs)}")
if len(unmatched_suburbs):
    print("Unmatched Greater Melbourne suburbs (first 20):")
    display(unmatched_suburbs[["suburb_id", "suburb_name"]].head(20))

# One row for each suburb and language, ready for the app's dynamic lookup.
language_counts = matched.melt(id_vars="suburb_id", value_vars=language_columns, var_name="language", value_name="speaker_count")
language_counts["speaker_count"] = language_counts["speaker_count"].astype("int64")

print("Detailed language/classification categories:", len(language_columns))
print("Language count rows:", len(language_counts))
display(language_counts.head())

TableBuilder suburbs matched: 572 / 572
Detailed language/classification categories: 503
Language count rows: 287716


,suburb_id,language,speaker_count
0,20002,"Northern European Languages, nfd",0
1,20003,"Northern European Languages, nfd",0
2,20011,"Northern European Languages, nfd",0
3,20015,"Northern European Languages, nfd",0
4,20017,"Northern European Languages, nfd",0


### 2.3 Rental costs and availability

In [67]:
# 1 - Clean and transform rental data

rent_tables = []

for property_type, sheet in ds_01.items():

    # Find the latest median rent column
    median_cols = [col for col in sheet.columns if str(sheet.loc[2, col]).strip().lower() == "median"]

    latest_col = median_cols[-1]
    latest_period = sheet.loc[1, latest_col]

    # Keep suburb and latest median rent
    rent = sheet.loc[3:, [1, latest_col]].copy()
    rent.columns = ["suburb", "median_weekly_rent"]

    # Convert rent to a number and remove empty records
    rent["median_weekly_rent"] = pd.to_numeric(rent["median_weekly_rent"], errors="coerce")

    rent = rent.dropna(subset=["suburb"])

    # The report is laid out in regional blocks, each closed off with a "Group Total"
    # row - that's a subtotal, not a suburb, so drop it before it gets treated as one.
    rent = rent[rent["suburb"].str.strip().str.lower() != "group total"]

    # Add dwelling type and reporting period
    rent["dwelling_type"] = property_type
    rent["period"] = latest_period

    rent_tables.append(rent)

# Combine all dwelling types into one table
rental_clean = pd.concat(rent_tables, ignore_index=True)

# Most rows aren't a single suburb but a combined precinct, e.g. "Carlton-Parkville".
# Keep the original label and also split it out so each suburb named in a precinct
# gets its own row carrying that precinct's rent figure.
rental_clean["precinct"] = rental_clean["suburb"]
rental_clean["suburb"] = rental_clean["suburb"].str.split("-")
rental_clean = rental_clean.explode("suburb", ignore_index=True)
rental_clean["suburb"] = rental_clean["suburb"].str.strip()


def normalize_rental_suburb_name(name):
    """Matching key just for lining up rental suburb names against `suburbs`.
    Only rental needs this extra cleanup: REIV sometimes writes compound names
    in a different order to ABS ("East Brunswick" vs "Brunswick East") and
    abbreviates "Mount" to "Mt", so on top of the usual lowercase/"(Vic.)" strip
    this also expands "Mt" and sorts the words."""
    name = normalize_suburb_name(name)
    name = re.sub(r"\bmt\b", "mount", name)
    return " ".join(sorted(name.split()))


# Drop the ABS "(Vic.)" tag from the suburbs list, then use that as the source of
# truth: wherever a rental suburb name matches one, overwrite it with the ABS
# spelling so both tables end up using exactly the same suburb names.
suburb_name_clean = suburbs["suburb_name"].str.replace(r"\s*\([^)]*\)\s*$", "", regex=True).str.strip()
canonical_lookup = dict(zip(suburb_name_clean.map(normalize_rental_suburb_name), suburb_name_clean))
rental_clean["suburb"] = rental_clean["suburb"].map(lambda name: canonical_lookup.get(normalize_rental_suburb_name(name), name))
rental_clean["suburb_join"] = rental_clean["suburb"].map(normalize_suburb_name)

print("Rental records:", len(rental_clean))
display(rental_clean.head())


# Clean LGA rental data using the same approach as suburb rents

lga_rent_tables = []

for property_type, sheet in ds_01_lga.items():


    # Find the latest median rent column
    median_cols = [col for col in sheet.columns if str(sheet.loc[2, col]).strip().lower() == "median"]

    latest_col = median_cols[-1]
    latest_period = sheet.loc[1, latest_col]

    # Keep LGA and latest median rent
    rent = sheet.loc[3:, [1, latest_col]].copy()
    rent.columns = ["lga_name", "median_weekly_rent"]

    rent["median_weekly_rent"] = pd.to_numeric(rent["median_weekly_rent"], errors="coerce")

    rent = rent.dropna(subset=["lga_name", "median_weekly_rent"])

    # Remove totals and clean names
    rent = rent[rent["lga_name"].str.strip().str.lower() != "group total"]


    rent["dwelling_type"] = property_type
    rent["period"] = latest_period

    lga_rent_tables.append(rent)

lga_rental_clean = pd.concat(lga_rent_tables, ignore_index=True)

# Create a matching key for LGA names
lga_rental_clean["lga_join"] = (lga_rental_clean["lga_name"].str.lower().str.strip())

print("LGA rental records:", len(lga_rental_clean))


# JG observation: about 1 in 12 rows here used to be "Group Total" - a region subtotal from
# the source spreadsheet, not an actual suburb - and were silently being counted as one.
# Dropped those, then split the combined precinct names (e.g. Carlton-Parkville) so each
# suburb in a precinct carries that precinct's rent figure. REIV also sometimes writes
# compound suburb names in a different order to ABS ("East Brunswick" vs "Brunswick East")
# or abbreviates "Mount" to "Mt" - normalize_rental_suburb_name (above) handles both, just
# for this table, and matched names get overwritten with the ABS spelling so the two tables
# agree. Altogether that lifts real suburb coverage from about 10% to around 29%, which
# looks like the practical ceiling: REIV only reports rent for ~146 named areas across the
# whole state, so most smaller Melbourne suburbs just aren't in this dataset at all - no
# amount of name-cleaning will recover that.

Rental records: 1505


,suburb,median_weekly_rent,dwelling_type,period,precinct,suburb_join
0,Albert Park,480.0,1 bedroom flat,Jun 2026,Albert Park-Middle Park-West St Kilda,albert park
1,Middle Park,480.0,1 bedroom flat,Jun 2026,Albert Park-Middle Park-West St Kilda,middle park
2,St Kilda West,480.0,1 bedroom flat,Jun 2026,Albert Park-Middle Park-West St Kilda,st kilda west
3,Armadale,478.0,1 bedroom flat,Jun 2026,Armadale,armadale
4,Carlton North,465.0,1 bedroom flat,Jun 2026,Carlton North,carlton north


LGA rental records: 480


In [68]:
# 1c - Rental trend history by suburb, with a Melbourne benchmark
# Uses the full quarterly history in the moving annual rent workbook (Mar 2000 onwards),
# not just the latest quarter, so the front end can chart each suburb against Melbourne.
# Relies on ds_01, suburbs, normalize_suburb_name, normalize_rental_suburb_name and
# canonical_lookup from the cells above.

METRO_REGIONS = ["Inner Melbourne", "Inner Eastern Melbourne", "Southern Melbourne", "Outer Western Melbourne", "North Western Melbourne", "North Eastern Melbourne", "Outer Eastern Melbourne", "South Eastern Melbourne", "Mornington Peninsula"]

trend_tables = []

for dwelling_type, sheet in ds_01.items():

    # Row 1 holds the quarter label and row 2 says whether the column is a Count or a Median.
    # Each quarter has a Count column followed by its Median column.
    periods = sheet.loc[1]
    measures = sheet.loc[2].astype(str).str.strip().str.lower()

    rows = sheet.loc[3:].copy()
    rows[0] = rows[0].ffill()                       # region is only written on its first row
    rows = rows.dropna(subset=[1])

    for col in sheet.columns[2:]:
        if measures[col] != "median":
            continue

        count_col = col - 1                          # the matching Count sits just before it
        quarter = pd.DataFrame({"region": rows[0].str.strip(), "precinct": rows[1].astype(str).str.strip(), "dwelling_type": dwelling_type, "period": pd.to_datetime(str(periods[col]).strip(), format="%b %Y") + pd.offsets.MonthEnd(0), "new_lease_count": pd.to_numeric(rows[count_col], errors="coerce"), "median_weekly_rent": pd.to_numeric(rows[col], errors="coerce")})
        trend_tables.append(quarter)

rental_history = pd.concat(trend_tables, ignore_index=True)
rental_history = rental_history[rental_history["region"].isin(METRO_REGIONS)]

# Melbourne benchmark: lease-weighted average of the metro region "Group Total" medians.
# The workbook has no single Melbourne figure, and region totals are true medians for
# each region, so weighting them by new leases is the closest available approximation.
group_totals = rental_history[rental_history["precinct"].str.lower() == "group total"].dropna(subset=["new_lease_count", "median_weekly_rent"])

melbourne_trend = (group_totals.assign(weighted=group_totals["median_weekly_rent"] * group_totals["new_lease_count"]).groupby(["dwelling_type", "period"], as_index=False).agg(weighted=("weighted", "sum"), leases=("new_lease_count", "sum")))
melbourne_trend["melbourne_avg_rent"] = (melbourne_trend["weighted"] / melbourne_trend["leases"]).round(0)
melbourne_trend = melbourne_trend[["dwelling_type", "period", "melbourne_avg_rent"]]

# Suburb history: drop the subtotals, then split precincts into suburbs the same way
# rental_clean does, so the two tables use identical suburb names.
rental_trend = rental_history[rental_history["precinct"].str.lower() != "group total"].copy()
rental_trend["suburb"] = rental_trend["precinct"].str.split("-")
rental_trend = rental_trend.explode("suburb", ignore_index=True)
rental_trend["suburb"] = rental_trend["suburb"].str.strip().map(lambda name: canonical_lookup.get(normalize_rental_suburb_name(name), name))
rental_trend["suburb_join"] = rental_trend["suburb"].map(normalize_suburb_name)

# Attach the ABS suburb_id and keep only our Greater Melbourne suburbs
rental_trend = rental_trend.merge(suburbs[["suburb_id", "suburb_name_join"]], left_on="suburb_join", right_on="suburb_name_join", how="inner").drop(columns="suburb_name_join")

# Add the benchmark and the gap to it, ready to chart
rental_trend = rental_trend.merge(melbourne_trend, on=["dwelling_type", "period"], how="left")
rental_trend["rent_vs_melbourne_pct"] = ((rental_trend["median_weekly_rent"] / rental_trend["melbourne_avg_rent"] - 1) * 100).round(1)

rental_trend = rental_trend[["suburb_id", "suburb", "precinct", "region", "dwelling_type", "period", "new_lease_count", "median_weekly_rent", "melbourne_avg_rent", "rent_vs_melbourne_pct"]].sort_values(["suburb_id", "dwelling_type", "period"]).reset_index(drop=True)

print("Rental trend rows:", len(rental_trend))
print("Suburbs with history:", rental_trend["suburb_id"].nunique())
print("Periods:", rental_trend["period"].min().date(), "to", rental_trend["period"].max().date())
display(rental_trend.tail())

# JG observation: the workbook has no Melbourne-wide row, only a "Group Total" per region,
# so melbourne_avg_rent is the new-lease-weighted average of the nine metro region medians.
# That's an approximation of a Melbourne median, not the real thing - label it "Melbourne
# average" in the UI. Suburbs inside a combined precinct (e.g. Carlton-Parkville) all share
# that precinct's history, same as in rental_clean.


Rental trend rows: 122430
Suburbs with history: 165
Periods: 2000-03-31 to 2026-06-30


,suburb_id,suburb,precinct,region,dwelling_type,period,new_lease_count,median_weekly_rent,melbourne_avg_rent,rent_vs_melbourne_pct
122425,22917,Yarraville,Yarraville-Seddon,Outer Western Melbourne,All properties,2025-06-30,1059.0,605.0,586.0,3.2
122426,22917,Yarraville,Yarraville-Seddon,Outer Western Melbourne,All properties,2025-09-30,1036.0,620.0,591.0,4.9
122427,22917,Yarraville,Yarraville-Seddon,Outer Western Melbourne,All properties,2025-12-31,1027.0,620.0,591.0,4.9
122428,22917,Yarraville,Yarraville-Seddon,Outer Western Melbourne,All properties,2026-03-31,1015.0,630.0,592.0,6.4
122429,22917,Yarraville,Yarraville-Seddon,Outer Western Melbourne,All properties,2026-06-30,1008.0,650.0,601.0,8.2


In [69]:
# 1b - June 2026 new rentals with precinct and LGA infill

PERIOD = "Jun 2026"

DWELLING_TYPES = ["1 bedroom flat", "2 bedroom flat", "3 bedroom flat", "2 bedroom house", "3 bedroom house", "4 bedroom house"]

# Extract June 2026 rental counts from Homes Victoria
rental_activity_tables = []

for dwelling_type in DWELLING_TYPES:

    sheet = ds_01[dwelling_type]

    count_cols = [col for col in sheet.columns if str(sheet.loc[1, col]).strip() == PERIOD and str(sheet.loc[2, col]).strip().lower() == "count"]

    assert len(count_cols) == 1, f"Expected one {PERIOD} count for {dwelling_type}"

    activity = sheet.loc[3:, [1, count_cols[0]]].copy()
    activity.columns = ["precinct", "new_rentals"]

    activity = activity.dropna(subset=["precinct"])
    activity = activity[activity["precinct"].astype(str).str.strip().str.lower() != "group total"].copy()

    activity["new_rentals"] = pd.to_numeric(activity["new_rentals"], errors="coerce")
    activity["dwelling_type"] = dwelling_type

    rental_activity_tables.append(activity)

rental_activity = pd.concat(rental_activity_tables, ignore_index=True)

assert not rental_activity.duplicated(["precinct", "dwelling_type"]).any(), "Duplicate rental precinct records"

# Reuse existing rental precinct-to-suburb matching
precinct_mapping = rental_clean[["precinct", "suburb_join", "dwelling_type"]].drop_duplicates()
precinct_mapping = precinct_mapping[precinct_mapping["dwelling_type"].isin(DWELLING_TYPES)].copy()

# Identify combined precincts before restricting to Greater Melbourne
precinct_mapping["precinct_members"] = precinct_mapping.groupby(["precinct", "dwelling_type"])["suburb_join"].transform("nunique")

# Attach suburb IDs
precinct_mapping = precinct_mapping.merge(
    suburbs[["suburb_id", "suburb_name_join"]],
    left_on="suburb_join", right_on="suburb_name_join",
    how="inner", validate="many_to_one")

# Attach published rental counts
suburb_activity = precinct_mapping.merge(
    rental_activity, on=["precinct", "dwelling_type"],
    how="left", validate="many_to_one")

# Retain the full published count for individual suburbs and combined precincts
suburb_activity["infilling_method"] = np.where(suburb_activity["precinct_members"] > 1, "precinct", "suburb")
suburb_activity["area_name"] = suburb_activity["precinct"]

suburb_activity = suburb_activity.dropna(subset=["new_rentals"]).copy()

# Prefer individual suburb counts over combined precinct counts
suburb_activity["priority"] = suburb_activity["infilling_method"].map({"suburb": 0, "precinct": 1})

suburb_activity = suburb_activity[suburb_activity["priority"].eq(suburb_activity.groupby(["suburb_id", "dwelling_type"])["priority"].transform("min"))].copy()

assert not suburb_activity.duplicated(["suburb_id", "dwelling_type"]).any(), "Duplicate suburb rental records"

# Include all suburbs and dwelling types
all_combinations = suburbs[["suburb_id"]].merge(pd.DataFrame({"dwelling_type": DWELLING_TYPES}), how="cross")

rental_availability = all_combinations.merge(
    suburb_activity[["suburb_id", "dwelling_type", "new_rentals", "infilling_method", "area_name"]],
    on=["suburb_id", "dwelling_type"], how="left", validate="one_to_one")

# Extract June 2026 LGA rental counts
lga_activity_tables = []

for dwelling_type in DWELLING_TYPES:

    sheet = ds_01_lga[dwelling_type]

    count_cols = [col for col in sheet.columns if str(sheet.loc[1, col]).strip() == PERIOD and str(sheet.loc[2, col]).strip().lower() == "count"]

    assert len(count_cols) == 1, f"Expected one {PERIOD} LGA count for {dwelling_type}"

    activity = sheet.loc[3:, [1, count_cols[0]]].copy()
    activity.columns = ["lga_name", "lga_new_rentals"]

    activity["lga_name"] = activity["lga_name"].astype("string").str.strip().replace(NAME_OVERRIDES)
    activity = activity[activity["lga_name"].isin(lga_matches["lga_name"])].copy()

    activity["lga_new_rentals"] = pd.to_numeric(activity["lga_new_rentals"], errors="coerce")
    activity["dwelling_type"] = dwelling_type

    lga_activity_tables.append(activity)

lga_rental_activity = pd.concat(lga_activity_tables, ignore_index=True)

assert not lga_rental_activity.duplicated(["lga_name", "dwelling_type"]).any(), "Duplicate LGA rental records"

# Assign each suburb to its primary LGA
# Use the LGA containing the largest proportion of the suburb
primary_lga = suburb_lga_mapping.sort_values(
    ["suburb_id", "lga_weighting", "lga_name"],
    ascending=[True, False, True]
).drop_duplicates("suburb_id")[["suburb_id", "lga_name"]]

# Attach LGA rental counts
rental_availability = rental_availability.merge(
    primary_lga, on="suburb_id",
    how="left", validate="many_to_one")

rental_availability = rental_availability.merge(
    lga_rental_activity, on=["lga_name", "dwelling_type"],
    how="left", validate="many_to_one")

# Infill missing suburb and precinct values using the full LGA count
missing = rental_availability["new_rentals"].isna() & rental_availability["lga_new_rentals"].notna()

rental_availability.loc[missing, "new_rentals"] = rental_availability.loc[missing, "lga_new_rentals"]
rental_availability.loc[missing, "infilling_method"] = "lga"
rental_availability.loc[missing, "area_name"] = rental_availability.loc[missing, "lga_name"]

# No further infilling - remaining missing values stay null

# Rank suburbs within each dwelling type (highest first)
rental_availability["rental_availability_ranking"] = rental_availability.groupby("dwelling_type")["new_rentals"].rank(ascending=False, method="min").astype("Int64")

# Score using 10th and 90th percentile boundaries
def rental_availability_score(series):
    """Scale rental counts 0–100 after clipping to within-type P10/P90 bounds."""
    valid = series.dropna()

    if valid.empty:
        return pd.Series(np.nan, index=series.index)
    if (valid == 0).all():
        return pd.Series(0.0, index=series.index).where(series.notna())

    p10, p90 = valid.quantile([0.1, 0.9])

    if p10 == p90:
        return pd.Series(100.0, index=series.index).where(series.notna())

    return ((series.clip(lower=p10, upper=p90) - p10) / (p90 - p10) * 100).round()

rental_availability["rental_availability_score"] = rental_availability.groupby("dwelling_type")["new_rentals"].transform(rental_availability_score).astype("Int64")

# Keep diagnostic information separately
rental_availability_diagnostics = rental_availability.copy()

# Final Silver table
rental_availability["new_rentals"] = rental_availability["new_rentals"].astype("Int64")

rental_availability = rental_availability[["suburb_id", "dwelling_type", "new_rentals", "rental_availability_ranking", "rental_availability_score", "infilling_method", "area_name"]]

print("Rental availability records:", len(rental_availability))
print(rental_availability["infilling_method"].value_counts(dropna=False))
display(rental_availability.head())

Rental availability records: 3432
infilling_method
lga         2325
precinct     620
suburb       345
NaN          142
Name: count, dtype: int64


,suburb_id,dwelling_type,new_rentals,rental_availability_ranking,rental_availability_score,infilling_method,area_name
0,20002,1 bedroom flat,599,17,100,precinct,Collingwood-Abbotsford
1,20002,2 bedroom flat,663,21,100,precinct,Collingwood-Abbotsford
2,20002,3 bedroom flat,77,308,44,precinct,Collingwood-Abbotsford
3,20002,2 bedroom house,73,60,97,precinct,Collingwood-Abbotsford
4,20002,3 bedroom house,40,559,0,precinct,Collingwood-Abbotsford


In [70]:
# 1c - Affordable rental lettings by LGA

affordable_tables = []

for dwelling_type, sheet in ds_affordable_lettings.items():

    # Find the June 2026 affordable lettings and percentage columns
    affordable_cols = [col for col in sheet.columns if str(sheet.loc[1, col]).strip() == PERIOD and str(sheet.loc[2, col]).strip().lower() == "affordable"]

    percent_cols = [col for col in sheet.columns if str(sheet.loc[1, col]).strip() == PERIOD and str(sheet.loc[2, col]).strip().lower() == "percent"]

    assert len(affordable_cols) == 1, f"Expected one {PERIOD} affordable count for {dwelling_type}"
    assert len(percent_cols) == 1, f"Expected one {PERIOD} affordable percentage for {dwelling_type}"

    # Keep LGA name, affordable lettings and affordable share
    affordable = sheet.loc[3:, [0, affordable_cols[0], percent_cols[0]]].copy()
    affordable.columns = ["lga_name", "affordable_lettings", "affordable_lettings_share"]

    affordable["lga_name"] = affordable["lga_name"].astype("string").str.strip().replace(NAME_OVERRIDES)
    affordable = affordable[affordable["lga_name"].isin(lga_matches["lga_name"])].copy()

    affordable["affordable_lettings"] = pd.to_numeric(affordable["affordable_lettings"], errors="coerce").astype("Int64")
    affordable["affordable_lettings_share"] = (pd.to_numeric(affordable["affordable_lettings_share"], errors="coerce") * 100).round(2)

    affordable["dwelling_type"] = dwelling_type
    affordable["period"] = PERIOD

    affordable_tables.append(affordable)

# Combine all dwelling types into one table
affordable_lettings = pd.concat(affordable_tables, ignore_index=True)

# Final Silver table
affordable_lettings = affordable_lettings[["lga_name", "dwelling_type", "period", "affordable_lettings", "affordable_lettings_share"]].sort_values(["lga_name", "dwelling_type"]).reset_index(drop=True)

assert not affordable_lettings.duplicated(["lga_name", "dwelling_type"]).any(), "Duplicate affordable letting records"
assert affordable_lettings["affordable_lettings_share"].dropna().between(0, 100).all(), "Invalid affordable lettings percentage"

print("Affordable letting records:", len(affordable_lettings))
print("LGAs:", affordable_lettings["lga_name"].nunique())
display(affordable_lettings.head())

Affordable letting records: 195
LGAs: 39


,lga_name,dwelling_type,period,affordable_lettings,affordable_lettings_share
0,Banyule,1 bedroom,Jun 2026,5,5.0
1,Banyule,2 bedroom,Jun 2026,24,6.8
2,Banyule,3 bedroom,Jun 2026,3,1.0
3,Banyule,4+ bedroom,Jun 2026,4,3.6
4,Banyule,All properties,Jun 2026,36,4.1


### 2.4 AMEP locations and rental diagnostics

In [71]:
# 11 - AMEP English class locations

amep = ds_11.copy()

if {"latitude", "longitude"}.issubset(amep.columns):
    # Ensure coordinates are numeric
    amep["latitude"] = pd.to_numeric(amep["latitude"], errors="coerce")
    amep["longitude"] = pd.to_numeric(amep["longitude"], errors="coerce")

    # Exclude records without valid coordinates
    amep = amep[amep["latitude"].between(-90, 90) & amep["longitude"].between(-180, 180)].copy()

    # Create spatial points
    amep = gpd.GeoDataFrame(amep, geometry=gpd.points_from_xy(amep["longitude"], amep["latitude"]), crs=4326).to_crs(7855)
else:
    # Silver copy stores the location as WKT geometry instead of latitude/longitude
    amep = gpd.GeoDataFrame(amep, geometry=gpd.GeoSeries.from_wkt(amep["geometry"]), crs=4326).to_crs(7855)

# Restrict to New Roots application geography
amep = amep[amep.intersects(melbourne_shape)].copy()

# Standardise output
amep_facilities = amep[["amep_id", "facility_name", "provider_name", "geometry"]].drop_duplicates(subset="amep_id")

amep_facilities = (amep_facilities.to_crs(4326).reset_index(drop=True))

print("AMEP locations retained:", len(amep_facilities))
display(amep_facilities.head())

AMEP locations retained: 126


,amep_id,facility_name,provider_name,geometry
0,AMEP_VIC_002,Wingate Avenue Community Centre,Melbourne Polytechnic (Melbourne AMEP),POINT (144.91942 -37.7795)
1,AMEP_VIC_005,Bellfield Community Hub,Melbourne Polytechnic (Melbourne AMEP),POINT (145.04013 -37.75498)
2,AMEP_VIC_007,Chisholm TAFE,Melbourne Polytechnic (Melbourne AMEP),POINT (145.34357 -38.04296)
3,AMEP_VIC_008,Box Hill Institute Elgar Road Campus,Melbourne Polytechnic (Melbourne AMEP),POINT (145.11492 -37.81554)
4,AMEP_VIC_009,Kangan Institute,Melbourne Polytechnic (Melbourne AMEP),POINT (144.91783 -37.67961)


In [72]:
# 12 - Services Australia service centres

services_australia = ds_12.copy()

# Ensure coordinates are numeric and valid
services_australia["latitude"] = pd.to_numeric(services_australia["latitude"], errors="coerce")
services_australia["longitude"] = pd.to_numeric(services_australia["longitude"], errors="coerce")
services_australia = services_australia[services_australia["latitude"].between(-90, 90) & services_australia["longitude"].between(-180, 180)].copy()

# Create spatial points
services_australia = gpd.GeoDataFrame(services_australia, geometry=gpd.points_from_xy(services_australia["longitude"], services_australia["latitude"]), crs=4326).to_crs(7855)

# Restrict to New Roots application geography
services_australia = services_australia[services_australia.intersects(melbourne_shape)].copy()

# Standardise output (centre names are unique, so they act as the ID)
services_australia_centres = services_australia.rename(columns={"name": "centre_name", "office_typ": "office_type", "address1": "address"})[["centre_name", "office_type", "address", "suburb", "postcode", "geometry"]].drop_duplicates(subset="centre_name")

services_australia_centres["address"] = services_australia_centres["address"].str.strip()

services_australia_centres = services_australia_centres.to_crs(4326).reset_index(drop=True)

print("Services Australia centres retained:", len(services_australia_centres))
display(services_australia_centres.head())

Services Australia centres retained: 36


,centre_name,office_type,address,suburb,postcode,geometry
0,Heidelberg Service Centre,"Centrelink Service Centre - Medicare, Centrelink",,Heidelberg,3084,POINT (145.06972 -37.75897)
1,Sunshine Service Centre,"Centrelink Service Centre - Medicare, Centrelink",,Sunshine,3020,POINT (144.82977 -37.77816)
2,Lilydale Service Centre,"Centrelink Service Centre - Medicare, Centrelink",,Lilydale,3140,POINT (145.34409 -37.75734)
3,Fountain Gate Service Centre,"Centrelink Service Centre - Medicare, Centrelink",,Narre Warren,3805,POINT (145.30631 -38.02255)
4,Cranbourne Service Centre,"Centrelink Service Centre - Medicare, Centrelink",,Cranbourne,3977,POINT (145.28454 -38.11246)


In [73]:
# 5b - Infill missing rental data

# Ensure all suburbs have a space for rental for each dwelling type

# Ensure every spatial suburb has a row for each dwelling type

all_combinations = (suburbs[["suburb_name_join"]].drop_duplicates().rename(columns={"suburb_name_join": "suburb_join"}).merge(pd.DataFrame({"dwelling_type": rental_clean["dwelling_type"].unique()}), how="cross"))

rental_clean = all_combinations.merge(rental_clean, on=["suburb_join", "dwelling_type"], how="left")

rental_clean["suburb"] = rental_clean["suburb"].fillna(rental_clean["suburb_join"])

# Fill missing suburb rents using weighted LGA rents, then proximity

# Label original rental values
rental_clean["infilling_method"] = "suburb"
rental_clean.loc[rental_clean["median_weekly_rent"].isna(), "infilling_method"] = pd.NA

# Link rental suburbs to suburb IDs
suburb_lookup = suburbs[["suburb_id", "suburb_name_join"]]

rental_clean = rental_clean.merge(suburb_lookup, left_on="suburb_join", right_on="suburb_name_join", how="left").drop(columns="suburb_name_join")

# Calculate weighted LGA rents
lga_mapping = suburb_lga_mapping.merge(lga_rental_clean, left_on="lga_name", right_on="lga_name", how="left")

lga_mapping["weighted_rent"] = (lga_mapping["median_weekly_rent"] * lga_mapping["lga_weighting"])

lga_rents = (lga_mapping.dropna(subset=["median_weekly_rent"]).groupby(["suburb_id", "dwelling_type"], as_index=False).agg(weighted_rent=("weighted_rent", "sum"), total_weight=("lga_weighting", "sum")))

# Account for missing data if suburb is part of more than one LGA

lga_rents["lga_rent"] = (lga_rents["weighted_rent"] / lga_rents["total_weight"])

rental_clean = rental_clean.merge(lga_rents[["suburb_id", "dwelling_type", "lga_rent"]], on=["suburb_id", "dwelling_type"], how="left")

# Fill missing values from LGA
missing = rental_clean["median_weekly_rent"].isna() & rental_clean["lga_rent"].notna()

rental_clean.loc[missing, "median_weekly_rent"] = rental_clean.loc[missing, "lga_rent"]
rental_clean.loc[missing, "infilling_method"] = "lga"

# Proximity fallback using original published suburb rents only
from scipy.spatial import cKDTree

points = suburbs[["suburb_id", "geometry"]].to_crs(7855).copy()
points["geometry"] = points.geometry.representative_point()
points["x"] = points.geometry.x
points["y"] = points.geometry.y

rental_clean = rental_clean.merge(points[["suburb_id", "x", "y"]], on="suburb_id", how="left")

for dwelling_type in rental_clean["dwelling_type"].unique():

    donors = rental_clean[(rental_clean["dwelling_type"] == dwelling_type) & (rental_clean["infilling_method"] == "suburb")].dropna(subset=["x", "y"])

    targets = rental_clean[(rental_clean["dwelling_type"] == dwelling_type) & rental_clean["median_weekly_rent"].isna()].dropna(subset=["x", "y"])

    if donors.empty or targets.empty:
        continue

    tree = cKDTree(donors[["x", "y"]].to_numpy())
    _, idx = tree.query(targets[["x", "y"]].to_numpy())

    rental_clean.loc[targets.index, "median_weekly_rent"] = (donors.iloc[idx]["median_weekly_rent"].to_numpy())
    rental_clean.loc[targets.index, "infilling_method"] = "proximity"

# Remove temporary columns
rental_clean = rental_clean.drop(columns=["suburb_id", "lga_rent", "x", "y"])

In [74]:
# Infilling method breakdown - Greater Melbourne only

melbourne_rentals = rental_clean[rental_clean["suburb_join"].isin(suburbs["suburb_name_join"])]

print("Rental records by infilling method:")
print(melbourne_rentals["infilling_method"].value_counts(dropna=False))

print("\nUnique suburbs by infilling method:")
print(melbourne_rentals.groupby("infilling_method")["suburb_join"].nunique())

print("\nTotal Greater Melbourne suburbs covered:", melbourne_rentals.loc[melbourne_rentals["median_weekly_rent"].notna(), "suburb_join"].nunique(), "/", suburbs["suburb_id"].nunique())

Rental records by infilling method:
infilling_method
lga          2800
suburb       1130
proximity      74
Name: count, dtype: int64

Unique suburbs by infilling method:
infilling_method
lga          420
proximity     59
suburb       165
Name: suburb_join, dtype: int64

Total Greater Melbourne suburbs covered: 572 / 572


### 2.5 Feature points, transport and health sources

In [75]:
# 6 - Vicmap features of interest
# Retain only the facility types required by New Roots.

feature_subtypes = {"place of worship": ["church", "gurdwara (sikh)", "mandir (hindu)", "monastry", "mosque", "synagogue", "vihara (buddhist)"], "community venue": ["community centre", "hall"], "cultural centre": ["library"], "health facility": ["community health centre"], "education centre": ["primary school", "secondary school", "primary/secondary school", "university", "tertiary institution"], "recreational resource": ["playground", "picnic site"]}

# Flatten approved type/subtype combinations
allowed_features = {(feature_type, subtype) for feature_type, subtypes in feature_subtypes.items() for subtype in subtypes}

# Reproject and restrict to application geography
facilities = ds_06.to_crs(7855).copy()
facilities = facilities[facilities.intersects(melbourne_shape)].copy()

# Apply the exact type/subtype filter
feature_pairs = list(zip(facilities["FTYPE"].astype("string").str.strip().str.lower(), facilities["FEATSUBTYP"].astype("string").str.strip().str.lower()))

facilities = facilities.loc[[pair in allowed_features for pair in feature_pairs]].copy()

# Create the Silver table
vicmap_facilities = facilities[["FEATURE_ID", "NAME", "FTYPE", "FEATSUBTYP", "geometry"]].rename(columns={"FEATURE_ID": "facility_id", "NAME": "facility_name", "FTYPE": "facility_type", "FEATSUBTYP": "facility_subtype"}).drop_duplicates(subset="facility_id")

vicmap_facilities = (vicmap_facilities.to_crs(4326).reset_index(drop=True))

assert len(vicmap_facilities) > 0, "No approved Vicmap facilities: check FTYPE/FEATSUBTYP values"
print("Relevant Vicmap facilities:", len(vicmap_facilities))

print(vicmap_facilities.groupby(["facility_type", "facility_subtype"]).size().to_string())

display(vicmap_facilities.head())

Relevant Vicmap facilities: 7836
facility_type          facility_subtype        
community venue        community centre             287
                       hall                         567
cultural centre        library                      148
education centre       primary school              1014
                       primary/secondary school     270
                       secondary school             444
                       tertiary institution          55
                       university                    41
health facility        community health centre       19
place of worship       church                       660
                       gurdwara (sikh)                7
                       mandir (hindu)                 6
                       monastry                       1
                       mosque                        34
                       synagogue                     32
                       vihara (buddhist)              8
recreational resource  

,facility_id,facility_name,facility_type,facility_subtype,geometry
0,634168,VICTORIA UNIVERSITY (TAFE DIVISION) - MELTON C...,education centre,tertiary institution,MULTIPOINT ((144.56531 -37.70713))
1,634198,HOLMESGLEN INSTITUTE OF TAFE - WAVERLEY CAMPUS,education centre,tertiary institution,MULTIPOINT ((145.15374 -37.88525))
2,634132,KANGAN BATMAN INSTITUTE OF TAFE - ESSENDON CAMPUS,education centre,tertiary institution,MULTIPOINT ((144.92324 -37.75738))
3,634113,CHISHOLM INSTITUTE OF TAFE - BERWICK CAMPUS,education centre,tertiary institution,MULTIPOINT ((145.34355 -38.04249))
4,634114,KANGAN BATMAN INSTITUTE OF TAFE - BROADMEADOWS...,education centre,tertiary institution,MULTIPOINT ((144.9159 -37.6777))


In [76]:
# 7 - Public transport stops
# Keep the stop ID, name, transport type and location.
stops = ds_07.to_crs(7855)
stops = stops[stops.intersects(melbourne_shape)]
stops = stops[["STOP_ID", "STOP_NAME", "MODE", "geometry"]].copy()
stops = stops.rename(columns={"STOP_ID": "stop_id", "STOP_NAME": "stop_name", "MODE": "transport_mode"})
stops["transport_mode"] = stops["transport_mode"].fillna("").str.lower()
stops = stops[stops["transport_mode"].str.contains("bus|tram|train")]
transport_stops = stops.drop_duplicates(subset="stop_id").to_crs(4326).reset_index(drop=True)

print("Relevant transport stops:", len(transport_stops))
display(transport_stops.head())

Relevant transport stops: 23779


,stop_id,stop_name,transport_mode,geometry
0,11212,Flinders Street Station,regional train,POINT (144.96627 -37.81809)
1,11213,Flinders Street Station,regional train,POINT (144.96649 -37.81814)
2,11214,Flinders Street Station,regional train,POINT (144.96652 -37.8182)
3,11215,Flinders Street Station,regional train,POINT (144.96656 -37.81829)
4,11216,Flinders Street Station,regional train,POINT (144.9666 -37.81833)


In [77]:
# 10 - GPs, pharmacies and hospitals
# Combine the three layers into one table with a facility type.
health_parts = []

for service, table in ds_10.items():
    health = table.to_crs(7855)
    health = health[health.intersects(melbourne_shape)].copy()
    name_col = next((c for c in health.columns if c.lower() in ["name", "facility_name", "service_name", "organisation_name"]), None)

    if name_col is None:
        health["facility_name"] = pd.NA
    else:
        health = health.rename(columns={name_col: "facility_name"})

    health["facility_type"] = service.replace("general_practices", "gp").replace("hospitals", "hospital").replace("pharmacies", "pharmacy")
    health_parts.append(health[["facility_type", "facility_name", "geometry"]])

assert health_parts, "No healthcare inputs found"
health_services = gpd.GeoDataFrame(pd.concat(health_parts, ignore_index=True), geometry="geometry", crs=7855)
health_services = health_services.drop_duplicates().to_crs(4326)

print("Health service locations:", len(health_services))
display(health_services.head())

Health service locations: 2580


,facility_type,facility_name,geometry
0,gp,Hillcrest Health Centre,POINT (144.9143 -37.68442)
1,gp,Coolaroo Clinic,POINT (144.93155 -37.66459)
2,gp,Beachside Doctors,POINT (144.96656 -38.33221)
3,gp,Peak Medical Gpc,POINT (144.7038 -37.88137)
4,gp,Belgrave Medical Clinic,POINT (145.34709 -37.90715)


### 2.6 Silver quality checks
Checks are run **before** exporting data. Review all failed checks and the LGA match report before treating Gold files as publication-ready. Some descriptive coverage checks are informational rather than hard failures.

In [78]:
clean_tables = {"suburbs": suburbs,
                "census_people": census_people,
                "language_counts": language_counts,
                "rental_clean": rental_clean,
                "crime_clean": crime_clean,
                "vicmap_facilities": vicmap_facilities,
                "transport_stops": transport_stops,
                "health_services": health_services,
                "rental_availability": rental_availability,
                "suburb_lga_mapping": suburb_lga_mapping,
                "amep_facilities": amep_facilities,
                "lga_spatial_cleaned": lga_spatial_cleaned,
                "rental_trend": rental_trend,
                "rental_trend_melbourne": melbourne_trend,
                "services_australia_centres": services_australia_centres,
                "affordable_lettings": affordable_lettings}

In [79]:
# Test all cleaned datasets before building the OBT

# Tables and their expected columns
tables = {"Suburbs": (suburbs, ["suburb_id", "suburb_name", "suburb_name_join", "suburb_area_km2"]), 
          "Census": (census_people, ["suburb_id", "population", "median_age"]), 
          "Languages": (language_counts, ["suburb_id", "language", "speaker_count"]), 
          "Rental": (rental_clean, ["suburb", "suburb_join", "median_weekly_rent", "dwelling_type", "period", "precinct"]), 
          "Crime": (crime_clean, ["suburb", "suburb_join", "total_incidents", "person_incidents", "property_incidents"]), 
          "Vicmap": (vicmap_facilities, ["facility_id", "facility_type", "facility_subtype"]), 
          "Transport": (transport_stops, ["stop_id", "transport_mode"]), 
          "Healthcare": (health_services, ["facility_type", "facility_name"]), 
          "Rental Trend": (rental_trend, ["suburb_id", "dwelling_type", "period", "melbourne_avg_rent"]), 
          "Rental Availability": (rental_availability, ["suburb_id", "dwelling_type", "new_rentals", "rental_availability_score", "infilling_method", "area_name"]),
          "Affordable Lettings": (affordable_lettings, ["lga_name", "dwelling_type", "affordable_lettings", "affordable_lettings_share"]),
          "AMEP": (amep_facilities, ["amep_id", "facility_name", "provider_name", "geometry"]), 
          "Suburb LGA Mapping": (suburb_lga_mapping, ["suburb_id", "lga_name", "lga_weighting"]),
          "Services Australia": (services_australia_centres, ["centre_name", "office_type", "address", "suburb", "postcode", "geometry"])}

results = []

for name, (df, columns) in tables.items():

    # Check the table contains records
    has_rows = len(df) > 0

    # Check all required columns exist
    has_columns = all(col in df.columns for col in columns)

    # Check for missing values in required columns
    missing_values = df[columns].isna().sum().sum() if has_columns else 0

    # Check spatial datasets have geometry and a coordinate system
    spatial_ok = True

    if isinstance(df, gpd.GeoDataFrame):
        spatial_ok = (df.crs is not None and df.geometry.notna().all() and df.crs.to_epsg() == 4326)

    # Determine result
    passed = has_rows and has_columns and spatial_ok

    # Additional checks for the replacement language table only.
    if name == "Languages" and has_columns:
        passed = passed and (missing_values == 0 and pd.api.types.is_numeric_dtype(df["speaker_count"]) and df["speaker_count"].ge(0).all() and not df.duplicated(["suburb_id", "language"]).any())

    results.append({"Dataset": name, "Rows": len(df), "Columns": len(df.columns), "Missing Values": missing_values, "Status": "PASS" if passed else "FAIL"})

# Display results
test_results = pd.DataFrame(results)

print(test_results)

print("Tests passed:", (test_results["Status"] == "PASS").sum(), "/", len(test_results))

# Structural checks that should stop the pipeline if violated.
assert not suburbs.duplicated("suburb_id").any(), "Duplicate suburb IDs"
assert not rental_availability.duplicated(["suburb_id", "dwelling_type"]).any(), "Duplicate rental activity rows"
assert np.isclose(suburb_lga_mapping.groupby("suburb_id")["lga_weighting"].sum(), 1, atol=1e-3).all(), "LGA weights must sum to one"
assert set(suburbs["suburb_id"]) <= set(suburb_lga_mapping["suburb_id"]), "Some suburbs have no LGA mapping"
assert (test_results["Status"] == "PASS").all(), "Silver table schema or geometry checks failed"

                Dataset    Rows  Columns  Missing Values Status
0               Suburbs     572        5               0   PASS
1                Census     572        3               0   PASS
2             Languages  287716        3               0   PASS
3                Rental    4004        7            5698   PASS
4                 Crime     648        7               0   PASS
5                Vicmap    7836        5               0   PASS
6             Transport   23779        4               0   PASS
7            Healthcare    2580        3               0   PASS
8          Rental Trend  122430       10               0   PASS
9   Rental Availability    3432        7             568   PASS
10  Affordable Lettings     195        5               0   PASS
11                 AMEP     126        4               0   PASS
12   Suburb LGA Mapping    1053        4               0   PASS
13   Services Australia      36        6               0   PASS
Tests passed: 14 / 14


In [80]:
# Tables that join back to suburbs on a shared key column
suburb_joins = {"census_people": ("suburb_id", "suburb_id"), "language_counts": ("suburb_id", "suburb_id"), "crime_clean": ("suburb_name_join", "suburb_join"), "rental_clean": ("suburb_name_join", "suburb_join"), "rental_availability": ("suburb_id", "suburb_id"), "suburb_lga_mapping": ("suburb_id", "suburb_id"), "rental_trend": ("suburb_id", "suburb_id")}

# Tables with no suburb key at all - these join to suburbs spatially (point-in-polygon)
spatial_joins = ["vicmap_facilities", "transport_stops", "health_services", "amep_facilities", "services_australia_centres"]

results = []
for name, (suburbs_col, child_col) in suburb_joins.items():
    child = clean_tables[name]

    # Only count suburbs with an available rental activity value
    if name == "rental_availability":
        child = child.dropna(subset=["new_rentals"])

    suburb_keys = set(suburbs[suburbs_col])
    child_keys = set(child[child_col].dropna())

    # Coverage: how many suburbs get a match in the child table
    suburbs_matched = suburbs[suburbs_col].isin(child_keys).sum()

    # Accuracy: how many child keys match a real suburb
    child_matched = len(child_keys & suburb_keys)

    results.append({"Table": name, "Join": f"suburbs.{suburbs_col} = {name}.{child_col}", "Suburbs covered": suburbs_matched, "Suburbs total": len(suburbs), "Coverage": f"{suburbs_matched / len(suburbs):.1%}", "Child matched": child_matched, "Child total": len(child_keys), "Accuracy": f"{child_matched / len(child_keys):.1%}" if child_keys else "n/a"})

# Same two checks, done spatially: which suburbs contain at least one point, and which
# points actually fall inside one of our suburb boundaries at all.
for name in spatial_joins:
    child = clean_tables[name]
    joined = gpd.sjoin(child[["geometry"]].reset_index(drop=True), suburbs[["suburb_id", "geometry"]], how="left", predicate="within")
    suburbs_matched = suburbs["suburb_id"].isin(joined["suburb_id"].dropna()).sum()
    child_matched = joined["suburb_id"].notna().sum()

    results.append({"Table": name, "Join": "point-in-polygon on geometry", "Suburbs covered": suburbs_matched, "Suburbs total": len(suburbs), "Coverage": f"{suburbs_matched / len(suburbs):.1%}", "Child matched": child_matched, "Child total": len(child), "Accuracy": f"{child_matched / len(child):.1%}" if len(child) else "n/a"})

join_quality = pd.DataFrame(results)
display(join_quality)

,Table,Join,Suburbs covered,Suburbs total,Coverage,Child matched,Child total,Accuracy
0,census_people,suburbs.suburb_id = census_people.suburb_id,572,572,100.0%,572,572,100.0%
1,language_counts,suburbs.suburb_id = language_counts.suburb_id,572,572,100.0%,572,572,100.0%
2,crime_clean,suburbs.suburb_name_join = crime_clean.suburb_...,569,572,99.5%,569,569,100.0%
3,rental_clean,suburbs.suburb_name_join = rental_clean.suburb...,572,572,100.0%,572,572,100.0%
4,rental_availability,suburbs.suburb_id = rental_availability.suburb_id,572,572,100.0%,572,572,100.0%
5,suburb_lga_mapping,suburbs.suburb_id = suburb_lga_mapping.suburb_id,572,572,100.0%,572,572,100.0%
6,rental_trend,suburbs.suburb_id = rental_trend.suburb_id,165,572,28.8%,165,165,100.0%
7,vicmap_facilities,point-in-polygon on geometry,509,572,89.0%,7707,7836,98.4%
8,transport_stops,point-in-polygon on geometry,466,572,81.5%,23777,23779,100.0%
9,health_services,point-in-polygon on geometry,362,572,63.3%,2579,2580,100.0%


### 2.7 Export Silver CSVs
These intermediate files remain available for inspection and reproducibility. Geometries are stored as WKT in longitude/latitude (EPSG:4326); Gold reads them back immediately below.

In [81]:
# Export the data if needed 


for name, df in clean_tables.items():
    file = cleaned / f"{name}.csv"
    df.to_csv(file, index=False)
    print(f"Saved: {file}")

Saved: C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Silver\suburbs.csv
Saved: C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Silver\census_people.csv
Saved: C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Silver\language_counts.csv
Saved: C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Silver\rental_clean.csv
Saved: C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Silver\crime_clean.csv
Saved: C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Silver\vicmap_facilities.csv
Saved: C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Silver\transport_stops.csv
Saved: C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Silver\health_services.csv
Saved: C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Silver\rental_availability.csv
Saved: C:\Mo

## 3. Gold — suburb scores and application-ready helper tables

The following steps are adapted from the uploaded Silver → Gold Python script. They read the freshly exported `cleaned/*.csv` files, compute ranking and percentile-capped category metrics, and write final outputs into `gold/`.

**Scoring convention:** Higher score means more favourable under the specified indicator; higher rent/crime/proximity distance are inverted. Percentile caps are the 10th and 90th values. Missing values are not silently converted to zero.

In [82]:
# The label map is supplied separately from the Bronze datasets.
language_map_file = find_input("new_roots_language_display_map.csv")

# Read Census language names and their display labels
language_display_names = pd.read_csv(language_map_file).set_index('language')['display_name']


def score(values, lower=False):
    """Return 0–100 percentile-capped scores; invert when smaller values are better.

    Missing data stays missing; all-zero groups score zero.
    """
    valid = values.dropna()
    if valid.empty:
        return pd.Series(np.nan, index=values.index)
    if (valid == 0).all():
        return pd.Series(0.0, index=values.index).where(values.notna())
    low, high = valid.quantile([0.1, 0.9])
    if high == low:
        return pd.Series(100.0, index=values.index).where(values.notna())
    result = (values.clip(low, high) - low) / (high - low) * 100
    return (100 - result if lower else result).round()


def add_metric(name, values, lower=False):
    """Attach source value, descending desirability rank, and score to Gold suburbs."""
    values = pd.Series(values, index=suburbs.index, dtype=float)
    suburbs[f'{name}_value'] = values
    suburbs[f'{name}_ranking'] = values.rank(ascending=lower, method='min').astype('Int64')
    suburbs[f'{name}_score'] = score(values, lower).astype('Int64')


def proximity(points):
    """Return each suburb centroid's nearest-point straight-line distance in km."""
    if points.empty:
        return np.full(len(suburbs), np.nan)
    point_xy = np.array([(p.x, p.y) for p in points.geometry])
    suburb_xy = np.array([(p.x, p.y) for p in suburbs.geometry.centroid])
    return cKDTree(point_xy).query(suburb_xy)[0] / 1000


def stop_density(points):
    """Count transport stops within each suburb per square kilometre."""
    joined = gpd.sjoin(points[['geometry']], suburbs[['suburb_id', 'geometry']], predicate='within')
    counts = joined.groupby('suburb_id').size()
    return suburbs.suburb_id.map(counts).fillna(0) / suburbs.suburb_area_km2


def read_points(filename):
    """Load a Silver CSV with EPSG:4326 WKT geometry and project to EPSG:7855."""
    data = pd.read_csv(cleaned / filename, dtype={'suburb_id': str})
    return gpd.GeoDataFrame(data, geometry=gpd.GeoSeries.from_wkt(data.geometry), crs=4326).to_crs(7855)

### 3.1 Gold setup, suburb centroid and affordability metrics

In [83]:
# Suburbs and census population (population needed for crime and language calculations)
suburbs = read_points('suburbs.csv').reset_index(drop=True)
people = pd.read_csv(cleaned / 'census_people.csv', dtype={'suburb_id': str})
suburbs = suburbs.merge(people[['suburb_id', 'population']], on='suburb_id', how='left', validate='one_to_one')
suburbs['suburb_area_km2'] = suburbs.geometry.area / 1_000_000


# Calculate suburb centroid coordinates
# EPSG:7855 for accurate centroid calculation
# EPSG:4326 for longitude and latitude

suburb_centroids = gpd.GeoSeries(suburbs.geometry.centroid, crs=7855).to_crs(4326)

suburbs['centroid_x_coord'] = suburb_centroids.x
suburbs['centroid_y_coord'] = suburb_centroids.y


# Rent metrics: each dwelling type is separate
rent = pd.read_csv(cleaned / 'rental_clean.csv')
dwellings = {'rent_1br_flat': '1 bedroom flat', 'rent_2br_flat': '2 bedroom flat', 'rent_3br_flat': '3 bedroom flat', 'rent_2br_house': '2 bedroom house', 'rent_3br_house': '3 bedroom house', 'rent_4br_house': '4 bedroom house'}
for name, dwelling in dwellings.items():
    dwelling_rent = rent.loc[rent.dwelling_type.eq(dwelling) & rent.median_weekly_rent.notna()]
    rates = dwelling_rent.groupby('suburb_join').median_weekly_rent.mean()
    add_metric(name, suburbs.suburb_name_join.map(rates), lower=True)

    # Carry through the origin of the rent for this dwelling type.
    # If multiple source rows contribute to an averaged rent, show all methods.
    methods = dwelling_rent.groupby('suburb_join').infilling_method.agg(lambda values: ', '.join(sorted(values.dropna().astype(str).unique())))
    suburbs[f'{name}_infilling_method'] = suburbs.suburb_name_join.map(methods)


# Rental availability: June 2026 new lettings
availability = pd.read_csv(cleaned / 'rental_availability.csv', dtype={'suburb_id': str})
assert not availability.duplicated(['suburb_id', 'dwelling_type']).any(), 'Duplicate rental availability records'

for name, dwelling in dwellings.items():
    dwelling_availability = availability.loc[availability.dwelling_type.eq(dwelling)].set_index('suburb_id')

    metric_name = name.replace('rent_', 'rental_availability_')

    add_metric(metric_name, suburbs.suburb_id.map(dwelling_availability['new_rentals']))

    suburbs[f'{metric_name}_infilling_method'] = suburbs.suburb_id.map(dwelling_availability['infilling_method'])
    suburbs[f'{metric_name}_area_name'] = suburbs.suburb_id.map(dwelling_availability['area_name'])

### 3.2 Safety, transport, healthcare and education

In [84]:
# Crime incidents per 1,000 residents
crime = pd.read_csv(cleaned / 'crime_clean.csv')
crime = crime.groupby('suburb_join')[['total_incidents', 'person_incidents', 'property_incidents']].sum()
for name, field in {'crime_per_1000': 'total_incidents', 'crime_person_per_1000': 'person_incidents', 'crime_property_per_1000': 'property_incidents'}.items():
    rate = suburbs.suburb_name_join.map(crime[field]) / suburbs.population.replace(0, np.nan) * 1000
    add_metric(name, rate.where(suburbs.population >= 500), lower=True)

# Transport
stops = read_points('transport_stops.csv')
bus_tram = stops[stops.transport_mode.isin(['metro bus', 'regional bus', 'metro tram'])]
trains = stops[stops.transport_mode.isin(['metro train', 'regional train'])]
add_metric('bus_tram_stop_density', stop_density(bus_tram))
add_metric('train_station_proximity', proximity(trains), lower=True)

# Healthcare
health = read_points('health_services.csv')
for name, facility in [('gp_proximity', 'gp'), ('hospital_proximity', 'hospital')]:
    add_metric(name, proximity(health[health.facility_type.eq(facility)]), lower=True)

# Schools and English classes
facilities = read_points('vicmap_facilities.csv')
facilities['geometry'] = facilities.geometry.centroid
add_metric('primary_school_proximity', proximity(facilities[facilities.facility_subtype.isin(['primary school', 'primary/secondary school'])]), lower=True)
add_metric('secondary_school_proximity', proximity(facilities[facilities.facility_subtype.isin(['secondary school', 'primary/secondary school'])]), lower=True)

english = read_points('amep_facilities.csv')
add_metric('english_class_proximity', proximity(english), lower=True)

### 3.3 Export Gold suburb OBT

In [ ]:
# Export one row per suburb (no population or median age)
columns = ['suburb_id', 'suburb_name', 'suburb_area_km2', 'centroid_x_coord', 'centroid_y_coord']
columns += [c for c in suburbs.columns if c.endswith(('_value', '_ranking', '_score', '_infilling_method','_area_name'))]
result = suburbs[columns].copy()
result['geometry'] = suburbs.to_crs(4326).geometry.to_wkt()
result.to_csv(gold_dir / 'new_roots_obt.csv', index=False)
print(f'Saved {len(result)} suburbs and {len(result.columns)} columns')

Saved 572 suburbs and 84 columns


### 3.4 Language community scores

In [86]:
# Language helper table: one row per suburb and language
languages = pd.read_csv(cleaned / 'language_counts.csv', dtype={'suburb_id': str})
languages = languages.merge(suburbs[['suburb_id', 'population']], on='suburb_id', how='left', validate='many_to_one')
languages['display_name'] = languages.language.map(language_display_names).fillna(languages.language)
languages['percent_spoken'] = languages.speaker_count / languages.population.replace(0, np.nan) * 100
languages['language_score_ranking'] = languages.groupby('language').percent_spoken.rank(ascending=False, method='min').astype('Int64')
languages['language_score'] = languages.groupby('language').percent_spoken.transform(score).astype('Int64')

# Language community size: total speakers per suburb
languages['speaker_count_ranking'] = languages.groupby('language').speaker_count.rank(ascending=False, method='min').astype('Int64')
languages['speaker_count_score'] = languages.groupby('language').speaker_count.transform(score).astype('Int64')

languages = languages[['suburb_id', 'language', 'display_name', 'percent_spoken', 'language_score', 'language_score_ranking', 'speaker_count', 'speaker_count_score', 'speaker_count_ranking']]
languages.to_csv(gold_dir / 'new_roots_languages.csv', index=False)
print(f'Saved {len(languages)} suburb-language rows')

Saved 287716 suburb-language rows


### 3.5 Community infrastructure proximity

In [87]:
# Community infrastructure helper table: nearest facility of each type
community_types = ['community centre', 'hall']
community = facilities[facilities.facility_subtype.isin(community_types)].copy()

infra_rows = []
for infra_type in sorted(community.facility_subtype.dropna().unique()):
    points = community[community.facility_subtype.eq(infra_type)]
    distances = pd.Series(proximity(points))
    infra_rows.append(pd.DataFrame({'suburb_id': suburbs.suburb_id, 'infra_type': infra_type, 'proximity': distances, 'infra_score': score(distances, lower=True).astype('Int64'), 'infra_score_ranking': distances.rank(ascending=True, method='min').astype('Int64')}))

infrastructure = pd.concat(infra_rows, ignore_index=True)
infrastructure.to_csv(gold_dir / 'new_roots_infrastructure.csv', index=False)
print(f'Saved {len(infrastructure)} suburb-infrastructure rows')

Saved 1144 suburb-infrastructure rows


### 3.6 Gold map features: Vicmap + health + AMEP

In [88]:
# Gold features table
# One row per facility, with coordinates in EPSG:4326

# 1. Map Vicmap types/subtypes to frontend categories
vicmap_type_mapping = {"place of worship": "places of worship", "community venue": "community services", "cultural centre": "community services", "health facility": "health", "education centre": "education", "recreational resource": "recreational services"}

# Clean display names for subtypes
subtype_mapping = {"gurdwara (sikh)": "gurdwara", "mandir (hindu)": "mandir", "monastry": "monastery", "vihara (buddhist)": "vihara"}

vicmap_features = facilities[["facility_type", "facility_subtype", "facility_name", "geometry"]].copy()

vicmap_features = vicmap_features.rename(columns={"facility_name": "name", "facility_type": "source_type", "facility_subtype": "feature_subtype"})

vicmap_features["feature_type"] = (vicmap_features["source_type"].str.strip().str.lower().map(vicmap_type_mapping))

vicmap_features["feature_subtype"] = (vicmap_features["feature_subtype"].str.strip().str.lower().replace(subtype_mapping))

# Respect New Roots' requested categorisation of community health centres.
vicmap_features.loc[vicmap_features["feature_subtype"].eq("community health centre"), "feature_type"] = "education"

vicmap_features = vicmap_features[["feature_type", "feature_subtype", "name", "geometry"]]


# Defence in depth: restrict the Gold Vicmap set even if Silver source expands.
allowed_gold_subtypes = {"places of worship": {"church", "gurdwara", "mandir", "monastery", "mosque", "synagogue", "vihara"}, "community services": {"library", "community centre", "hall"}, "education": {"primary school", "secondary school", "primary/secondary school", "university", "tertiary institution", "community health centre"}, "recreational services": {"playground", "picnic site"}}
vicmap_features = vicmap_features[[subtype in allowed_gold_subtypes.get(category, set()) for category, subtype in zip(vicmap_features["feature_type"], vicmap_features["feature_subtype"])]].copy()

# 2. Healthcare services
# Hospitals, GPs and pharmacies from separate health layers

health_type_mapping = {"gp": "general practice", "hospital": "hospital", "pharmacy": "pharmacy"}

health_features = health[["facility_type", "facility_name", "geometry"]].copy()

health_features["feature_type"] = "health"

health_features["feature_subtype"] = (health_features["facility_type"].map(health_type_mapping))

health_features = health_features.rename(columns={"facility_name": "name"})

health_features = health_features[health_features["feature_subtype"].isin({"general practice", "hospital", "pharmacy"})].copy()

health_features = health_features[["feature_type", "feature_subtype", "name", "geometry"]]


# 3. AMEP English class facilities

amep_features = english[["facility_name", "geometry"]].copy()

amep_features = amep_features.rename(columns={"facility_name": "name"})

amep_features["feature_type"] = "english classes"
amep_features["feature_subtype"] = "amep class"

amep_features = amep_features[["feature_type", "feature_subtype", "name", "geometry"]]



# 4. Services Australia service centres

services_australia_points = read_points('services_australia_centres.csv')

services_features = services_australia_points[["centre_name", "geometry"]].copy()

services_features = services_features.rename(columns={"centre_name": "name"})

services_features["feature_type"] = "government services"
services_features["feature_subtype"] = "services australia centre"

services_features = services_features[["feature_type", "feature_subtype", "name", "geometry"]]


# 5. Combine the four sources

features = gpd.GeoDataFrame(pd.concat([vicmap_features, health_features, amep_features, services_features], ignore_index=True), geometry="geometry", crs=7855)


# 5. Clean values and remove invalid geometries

features["name"] = (features["name"].astype("string").str.strip())

features["feature_type"] = (features["feature_type"].astype("string").str.strip().str.lower())

features["feature_subtype"] = (features["feature_subtype"].astype("string").str.strip().str.lower())

features = features[features["feature_type"].notna() & features["feature_subtype"].notna() & features["geometry"].notna() & ~features["geometry"].is_empty].copy()


# 6. Convert geometries to points
# Retains points and creates representative points for polygons

features["geometry"] = (features.geometry.representative_point())

# Convert to longitude / latitude
features = features.to_crs(4326)

features["x_coord"] = features.geometry.x
features["y_coord"] = features.geometry.y


# 7. Final columns and duplicate removal

features = features[["feature_type", "feature_subtype", "name", "x_coord", "y_coord"]].drop_duplicates().reset_index(drop=True)


# 8. Validate final dataset

assert features["x_coord"].between(-180, 180).all()
assert features["y_coord"].between(-90, 90).all()

allowed_types = {"places of worship", "community services", "health", "education", "recreational services", "english classes", "government services"}

assert set(features["feature_type"]) <= allowed_types, ("Unexpected feature type in Gold dataset")


# 9. Export

features.to_csv(gold_dir / "new_roots_features.csv", index=False)

print(f"Saved {len(features)} features")

print(features.groupby(["feature_type", "feature_subtype"]).size().to_string())

Saved 10577 features
feature_type           feature_subtype          
community services     community centre              287
                       hall                          567
                       library                       148
education              community health centre        19
                       primary school               1014
                       primary/secondary school      270
                       secondary school              443
                       tertiary institution           55
                       university                     41
english classes        amep class                    126
government services    services australia centre      36
health                 general practice             1546
                       hospital                       84
                       pharmacy                      950
places of worship      church                        660
                       gurdwara                        7
                  

### 3.7 Gold rent cost trend

In [89]:
# 3.7 Gold rent cost trend
# One row per suburb, dwelling type and quarter, ready to chart against the Melbourne average.
# Reads the Silver rental_trend table and adds year-on-year change for the suburb and Melbourne.

rent_trend = pd.read_csv(cleaned / "rental_trend.csv", dtype={"suburb_id": str}, parse_dates=["period"])

# Use the same suburb names as the OBT
rent_trend = rent_trend.merge(suburbs[["suburb_id", "suburb_name"]], on="suburb_id", how="left", validate="many_to_one")

# Year-on-year change: compare each quarter with the same quarter a year earlier
last_year = rent_trend[["suburb_id", "dwelling_type", "period", "median_weekly_rent", "melbourne_avg_rent"]].copy()
last_year["period"] = last_year["period"] + pd.DateOffset(years=1) + pd.offsets.MonthEnd(0)
last_year = last_year.rename(columns={"median_weekly_rent": "rent_last_year", "melbourne_avg_rent": "melbourne_last_year"})

rent_trend = rent_trend.merge(last_year, on=["suburb_id", "dwelling_type", "period"], how="left", validate="one_to_one")
rent_trend["rent_change_1yr_pct"] = ((rent_trend["median_weekly_rent"] / rent_trend["rent_last_year"] - 1) * 100).round(1)
rent_trend["melbourne_change_1yr_pct"] = ((rent_trend["melbourne_avg_rent"] / rent_trend["melbourne_last_year"] - 1) * 100).round(1)

# Final column order; period as YYYY-MM-DD text so the front end doesn't have to parse timestamps
rent_trend["period"] = rent_trend["period"].dt.strftime("%Y-%m-%d")
rent_trend = rent_trend[["suburb_id", "suburb_name", "precinct", "region", "dwelling_type", "period", "new_lease_count", "median_weekly_rent", "melbourne_avg_rent", "rent_vs_melbourne_pct", "rent_change_1yr_pct", "melbourne_change_1yr_pct"]].sort_values(["suburb_id", "dwelling_type", "period"]).reset_index(drop=True)

# Checks before saving
assert not rent_trend.duplicated(["suburb_id", "dwelling_type", "period"]).any(), "Duplicate rent trend rows"
assert rent_trend["suburb_id"].isin(suburbs["suburb_id"]).all(), "Rent trend has suburbs outside the OBT"
assert rent_trend["melbourne_avg_rent"].notna().all(), "Missing Melbourne benchmark values"

rent_trend.to_csv(gold_dir / "new_roots_rent_trend.csv", index=False)
print(f"Saved {len(rent_trend)} rent trend rows for {rent_trend['suburb_id'].nunique()} suburbs " f"({rent_trend['period'].min()} to {rent_trend['period'].max()})")
display(rent_trend.tail())


Saved 122430 rent trend rows for 165 suburbs (2000-03-31 to 2026-06-30)


,suburb_id,suburb_name,precinct,region,dwelling_type,period,new_lease_count,median_weekly_rent,melbourne_avg_rent,rent_vs_melbourne_pct,rent_change_1yr_pct,melbourne_change_1yr_pct
122425,22917,Yarraville,Yarraville-Seddon,Outer Western Melbourne,All properties,2025-06-30,1059.0,605.0,586.0,3.2,2.5,5.8
122426,22917,Yarraville,Yarraville-Seddon,Outer Western Melbourne,All properties,2025-09-30,1036.0,620.0,591.0,4.9,4.2,3.9
122427,22917,Yarraville,Yarraville-Seddon,Outer Western Melbourne,All properties,2025-12-31,1027.0,620.0,591.0,4.9,3.3,3.0
122428,22917,Yarraville,Yarraville-Seddon,Outer Western Melbourne,All properties,2026-03-31,1015.0,630.0,592.0,6.4,5.0,1.7
122429,22917,Yarraville,Yarraville-Seddon,Outer Western Melbourne,All properties,2026-06-30,1008.0,650.0,601.0,8.2,7.4,2.6


In [90]:
# 3.8 Gold affordable rental lettings
# One row per LGA and dwelling type

affordable_lettings = pd.read_csv(cleaned / "affordable_lettings.csv")

# Checks before saving
assert not affordable_lettings.duplicated(["lga_name", "dwelling_type"]).any(), "Duplicate affordable letting records"

affordable_lettings.to_csv(gold_dir / "new_roots_affordable_lettings.csv", index=False)

print(f"Saved {len(affordable_lettings)} affordable letting records for {affordable_lettings['lga_name'].nunique()} LGAs")
display(affordable_lettings.head())

Saved 195 affordable letting records for 39 LGAs


,lga_name,dwelling_type,period,affordable_lettings,affordable_lettings_share
0,Banyule,1 bedroom,Jun 2026,5,5.0
1,Banyule,2 bedroom,Jun 2026,24,6.8
2,Banyule,3 bedroom,Jun 2026,3,1.0
3,Banyule,4+ bedroom,Jun 2026,4,3.6
4,Banyule,All properties,Jun 2026,36,4.1


## 4. Final checks and output inventory
The last cell checks that the expected Gold files exist, that the OBT has one row per suburb, that map features use only approved categories, and that longitudes/latitudes are valid. See the README for interpretation and known limitations.

In [91]:
expected_gold = ["new_roots_obt.csv", "new_roots_languages.csv", "new_roots_infrastructure.csv", "new_roots_features.csv", "new_roots_rent_trend.csv", "new_roots_affordable_lettings.csv"]
for filename in expected_gold:
    output_path = gold_dir / filename
    assert output_path.is_file(), f"Gold output missing: {output_path}"
    print(f"{output_path}: {output_path.stat().st_size / 1024:,.1f} KB")

assert result["suburb_id"].is_unique, "Gold OBT must have one row per suburb"
assert result[["centroid_x_coord", "centroid_y_coord"]].notna().all().all()
assert result["centroid_x_coord"].between(-180, 180).all()
assert result["centroid_y_coord"].between(-90, 90).all()
assert not features.empty and not features[["feature_type", "feature_subtype", "x_coord", "y_coord"]].isna().any().any()
print("Gold validation complete.")

C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Gold\new_roots_obt.csv: 6,486.5 KB
C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Gold\new_roots_languages.csv: 14,661.6 KB
C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Gold\new_roots_infrastructure.csv: 48.6 KB
C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Gold\new_roots_features.csv: 899.2 KB
C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Gold\new_roots_rent_trend.csv: 14,127.7 KB
C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\data\Gold\new_roots_affordable_lettings.csv: 7.4 KB
Gold validation complete.
